# 04 Model Comparison
Compare Logistic Regression, Random Forest, and XGBoost on the same split.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path('..').resolve()))
from src.model import build_candidate_models, train_pipeline, predict_with_scores
from src.preprocessing import add_hit_label, make_preprocessor, temporal_split
from src.evaluation import classification_metrics

df = pd.read_csv('../data/raw/spotify.csv')
df.columns = [c.lower() for c in df.columns]

if 'hit' not in df.columns:
    popularity_col = 'popularity' if 'popularity' in df.columns else 'track_popularity'
    df = add_hit_label(df, popularity_col=popularity_col, threshold=70)

target_col = 'hit'
time_col = 'first_week' if 'first_week' in df.columns else None
if time_col is None:
    df['__time_proxy__'] = range(len(df))
    time_col = '__time_proxy__'

feature_df = df.copy()
x_train, x_test, y_train, y_test = temporal_split(feature_df, time_col=time_col, target_col=target_col)

num_cols = x_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in x_train.columns if c not in num_cols]
preprocessor = make_preprocessor(num_cols, cat_cols)

candidate_models = build_candidate_models()
list(candidate_models.keys())

['logistic_regression', 'random_forest', 'xgboost']

In [2]:
results = []
trained_models = {}

for model_name, estimator in candidate_models.items():
    pipeline = train_pipeline(preprocessor, estimator, x_train, y_train)
    y_pred, y_score = predict_with_scores(pipeline, x_test)
    metrics = classification_metrics(y_test, y_pred, y_score)
    metrics['model'] = model_name
    results.append(metrics)
    trained_models[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False)
results_df

,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,model
2,1.000000,1.000000,1.000000,1.000000,1.000000,xgboost
1,0.994405,0.996235,0.984257,0.990124,0.999936,random_forest
0,0.981350,0.951456,0.988718,0.968784,0.999791,logistic_regression


In [3]:
results_path = Path('../reports/results/model_comparison.csv')
results_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_path, index=False)
print(f'Saved comparison results to: {results_path}')

Saved comparison results to: ..\reports\results\model_comparison.csv
